In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap

# Set style for better visualization
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.family'] = 'DejaVu Sans'
%matplotlib inline

In [ ]:
# Load the music data
def load_data():
    try:
        # Try to load preprocessed data first
        df = pd.read_csv('music_data_preprocessed.csv')
        print("Loaded preprocessed data")
    except:
        # If not available, load and preprocess the raw data
        df = pd.read_csv('music-data.csv', header=None)
        print("Loaded raw data")
        
        # Assuming the structure based on files seen in the repository
        columns = ['artist', 'title', 'popularity', 'danceability', 'energy',
                  'key', 'loudness', 'mode', 'speechiness', 'acousticness',
                  'instrumentalness', 'valence', 'tempo', 'duration_ms',
                  'time_signature', 'year']
        
        df.columns = columns
    
    # Ensure year is int
    df['year'] = df['year'].astype(int)
    
    # Add decade column for grouping
    df['decade'] = (df['year'] // 10) * 10
    df['decade_label'] = df['decade'].astype(str) + 's'
    
    # Define key mapping (0=C, 1=C#, 2=D, etc.)
    key_mapping = {
        0: 'C', 1: 'C#/Db', 2: 'D', 3: 'D#/Eb', 
        4: 'E', 5: 'F', 6: 'F#/Gb', 7: 'G', 
        8: 'G#/Ab', 9: 'A', 10: 'A#/Bb', 11: 'B'
    }
    
    # Map keys to their names
    df['key_name'] = df['key'].map(key_mapping)
    
    # Map modes (0 = minor, 1 = major)
    mode_mapping = {0: 'Minor', 1: 'Major'}
    df['mode_name'] = df['mode'].map(mode_mapping)
    
    return df

# Load the data
df = load_data()
print(f"Loaded {len(df)} songs spanning from {df['year'].min()} to {df['year'].max()}")
df.head()

In [ ]:
def analyze_distributions(df):
    """Analyze if attributes like tempo, key, valence have recognizable distributions"""
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    
    # Analyze tempo distribution
    ax = axes[0, 0]
    sns.histplot(df['tempo'], kde=True, ax=ax, bins=30)
    ax.set_title('Tempo Distribution', fontsize=16)
    ax.set_xlabel('Tempo (BPM)', fontsize=12)
    
    # Add distribution fit
    x = np.linspace(df['tempo'].min(), df['tempo'].max(), 100)
    params = stats.norm.fit(df['tempo'])
    pdf_fitted = stats.norm.pdf(x, *params)
    ax.plot(x, pdf_fitted * len(df) * (df['tempo'].max() - df['tempo'].min()) / 30, 
            'r-', linewidth=2, label=f'Normal Fit\nμ={params[0]:.2f}, σ={params[1]:.2f}')
    
    # Test for normality
    k2, p = stats.normaltest(df['tempo'])
    ax.text(0.95, 0.95, f"Normality test p-value: {p:.4f}\n{'Normal' if p > 0.05 else 'Not normal'}", 
            transform=ax.transAxes, ha='right', va='top', 
            bbox=dict(facecolor='white', alpha=0.8))
    ax.legend()
    
    # Analyze valence distribution
    ax = axes[0, 1]
    sns.histplot(df['valence'], kde=True, ax=ax, bins=20)
    ax.set_title('Valence Distribution (Positivity/Mood)', fontsize=16)
    ax.set_xlabel('Valence (0-1)', fontsize=12)
    
    # Test for specific distributions (like beta which is common for 0-1 data)
    x = np.linspace(0, 1, 100)
    params = stats.beta.fit(df['valence'])
    pdf_fitted = stats.beta.pdf(x, *params)
    ax.plot(x, pdf_fitted * len(df) * (1 / 20), 
            'r-', linewidth=2, label=f'Beta Distribution Fit\nα={params[0]:.2f}, β={params[1]:.2f}')
    ax.legend()
    
    # Key distribution
    ax = axes[1, 0]
    key_counts = df['key_name'].value_counts().reindex(
        [key_mapping[i] for i in range(12)]  # Reindex to show all keys in order
    )
    sns.barplot(x=key_counts.index, y=key_counts.values, ax=ax)
    ax.set_title('Key Distribution', fontsize=16)
    ax.set_xlabel('Key', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.tick_params(axis='x', rotation=45)
    
    # Test for uniformity
    chi2, p = stats.chisquare(key_counts)
    ax.text(0.95, 0.95, f"Uniformity test p-value: {p:.4f}\n{'Uniform' if p > 0.05 else 'Not uniform'}", 
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(facecolor='white', alpha=0.8))
    
    # Energy distribution
    ax = axes[1, 1]
    sns.histplot(df['energy'], kde=True, ax=ax, bins=20)
    ax.set_title('Energy Distribution', fontsize=16)
    ax.set_xlabel('Energy (0-1)', fontsize=12)
    
    # Test for beta distribution
    params = stats.beta.fit(df['energy'])
    x = np.linspace(0, 1, 100)
    pdf_fitted = stats.beta.pdf(x, *params)
    ax.plot(x, pdf_fitted * len(df) * (1 / 20), 
            'r-', linewidth=2, label=f'Beta Distribution Fit\nα={params[0]:.2f}, β={params[1]:.2f}')
    ax.legend()
    
    plt.tight_layout()
    
# Run the distribution analysis
analyze_distributions(df)

In [ ]:
def analyze_time_trends(df):
    """Evaluate if tempo, keys, or other attributes show trends over time"""
    fig, axes = plt.subplots(2, 2, figsize=(20, 15))
    
    # Filter for realistic years (remove outliers)
    year_min = 1950
    year_max = 2025
    df_filtered = df[(df['year'] >= year_min) & (df['year'] <= year_max)]
    
    # Group by year and calculate mean tempo
    tempo_by_year = df_filtered.groupby('year')['tempo'].mean().reset_index()
    
    # Tempo trends
    ax = axes[0, 0]
    sns.regplot(x='year', y='tempo', data=tempo_by_year, ax=ax, scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})
    ax.set_title('Average Tempo Over Time', fontsize=16)
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Average Tempo (BPM)', fontsize=12)
    
    # Calculate regression statistics
    slope, intercept, r_value, p_value, std_err = stats.linregress(tempo_by_year['year'], tempo_by_year['tempo'])
    ax.text(0.05, 0.95, f"Slope: {slope:.4f} BPM/year\nR²: {r_value**2:.4f}\np-value: {p_value:.4f}", 
            transform=ax.transAxes, ha='left', va='top',
            bbox=dict(facecolor='white', alpha=0.8))
    
    # Energy trends
    ax = axes[0, 1]
    energy_by_year = df_filtered.groupby('year')['energy'].mean().reset_index()
    sns.regplot(x='year', y='energy', data=energy_by_year, ax=ax, scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})
    ax.set_title('Average Energy Over Time', fontsize=16)
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Average Energy (0-1)', fontsize=12)
    
    # Calculate regression statistics
    slope, intercept, r_value, p_value, std_err = stats.linregress(energy_by_year['year'], energy_by_year['energy'])
    ax.text(0.05, 0.95, f"Slope: {slope:.6f} per year\nR²: {r_value**2:.4f}\np-value: {p_value:.4f}", 
            transform=ax.transAxes, ha='left', va='top',
            bbox=dict(facecolor='white', alpha=0.8))
    
    # Key popularity by decade
    ax = axes[1, 0]
    # Get top 5 keys for each decade
    decade_key_counts = df_filtered.groupby(['decade', 'key_name']).size().reset_index(name='count')
    pivot_data = decade_key_counts.pivot(index='decade', columns='key_name', values='count')
    pivot_data = pivot_data.fillna(0)
    
    # Normalize by decade
    for decade in pivot_data.index:
        total = pivot_data.loc[decade].sum()
        pivot_data.loc[decade] = pivot_data.loc[decade] / total * 100
    
    # Plot heatmap
    sns.heatmap(pivot_data, ax=ax, cmap='viridis', annot=True, fmt='.1f', cbar_kws={'label': 'Percentage (%)'})
    ax.set_title('Key Popularity by Decade (% of songs)', fontsize=16)
    ax.set_ylabel('Decade', fontsize=12)
    ax.set_xlabel('Key', fontsize=12)
    
    # Valence trends
    ax = axes[1, 1]
    valence_by_year = df_filtered.groupby('year')['valence'].mean().reset_index()
    sns.regplot(x='year', y='valence', data=valence_by_year, ax=ax, scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})
    ax.set_title('Average Valence (Positivity) Over Time', fontsize=16)
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Average Valence (0-1)', fontsize=12)
    
    # Calculate regression statistics
    slope, intercept, r_value, p_value, std_err = stats.linregress(valence_by_year['year'], valence_by_year['valence'])
    ax.text(0.05, 0.95, f"Slope: {slope:.6f} per year\nR²: {r_value**2:.4f}\np-value: {p_value:.4f}", 
            transform=ax.transAxes, ha='left', va='top',
            bbox=dict(facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    
# Run the time trends analysis
analyze_time_trends(df)

In [ ]:
def create_heatmaps_and_timelines(df):
    """Create heatmaps and timeline visualizations"""
    # Filter for realistic years and ensure we have enough data points per decade
    year_min = 1950
    year_max = 2025
    df_filtered = df[(df['year'] >= year_min) & (df['year'] <= year_max)]
    
    # Group by decade and calculate average values for multiple attributes
    decade_attributes = df_filtered.groupby('decade_label')[
        ['tempo', 'danceability', 'energy', 'valence', 'acousticness', 'speechiness']
    ].mean().reset_index()
    
    # Prepare data for heatmap
    pivot_data = decade_attributes.set_index('decade_label')
    
    # Normalize each attribute column to 0-1 range for fair comparison
    for col in pivot_data.columns:
        min_val = pivot_data[col].min()
        max_val = pivot_data[col].max()
        if max_val > min_val:  # Prevent division by zero
            pivot_data[col] = (pivot_data[col] - min_val) / (max_val - min_val)
    
    # Sort by decades chronologically
    sorted_decades = sorted(pivot_data.index, key=lambda x: int(x[:-1]))
    pivot_data = pivot_data.reindex(sorted_decades)
    
    # Create heatmap
    plt.figure(figsize=(14, 8))
    ax = sns.heatmap(pivot_data.T, annot=True, fmt='.2f', cmap='viridis',
                   cbar_kws={'label': 'Normalized Value'})
    ax.set_title('Musical Attribute Trends by Decade (Normalized Values)', fontsize=16)
    ax.set_ylabel('Attribute', fontsize=14)
    ax.set_xlabel('Decade', fontsize=14)
    
    plt.tight_layout()
    plt.show()
    
    # Create mode (major/minor) timeline
    plt.figure(figsize=(16, 6))
    mode_by_year = df_filtered.groupby(['year', 'mode_name']).size().reset_index(name='count')
    mode_pivot = mode_by_year.pivot(index='year', columns='mode_name', values='count').fillna(0)
    
    # Calculate percentage
    mode_pivot['total'] = mode_pivot['Major'] + mode_pivot['Minor']
    mode_pivot['Major_pct'] = mode_pivot['Major'] / mode_pivot['total'] * 100
    mode_pivot['Minor_pct'] = mode_pivot['Minor'] / mode_pivot['total'] * 100
    
    # Create a stacked area chart
    plt.stackplot(mode_pivot.index, 
                  mode_pivot['Major_pct'], 
                  mode_pivot['Minor_pct'],
                  labels=['Major', 'Minor'],
                  colors=['#2ecc71', '#e74c3c'],  # Green for Major, Red for Minor
                  alpha=0.7)
    
    plt.title('Proportion of Major vs Minor Keys Over Time', fontsize=16)
    plt.xlabel('Year', fontsize=14)
    plt.ylabel('Percentage (%)', fontsize=14)
    plt.legend(loc='upper right', fontsize=12)
    plt.xlim(year_min, year_max)
    plt.ylim(0, 100)
    
    # Add trend lines for major and minor
    major_trend = np.polyfit(mode_pivot.index, mode_pivot['Major_pct'], 1)
    minor_trend = np.polyfit(mode_pivot.index, mode_pivot['Minor_pct'], 1)
    
    plt.plot(mode_pivot.index, np.polyval(major_trend, mode_pivot.index), 
             'g--', linewidth=2, label='Major trend')
    plt.plot(mode_pivot.index, np.polyval(minor_trend, mode_pivot.index), 
             'r--', linewidth=2, label='Minor trend')
    
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Create combined timeline visualization for tempo and valence
    fig, ax1 = plt.subplots(figsize=(16, 8))
    
    # Group by decade for smoother visualization
    tempo_by_decade = df_filtered.groupby('decade')['tempo'].mean()
    valence_by_decade = df_filtered.groupby('decade')['valence'].mean()
    
    color1 = '#3498db'  # Blue
    color2 = '#e74c3c'  # Red
    
    # Plot tempo
    ax1.set_xlabel('Decade', fontsize=14)
    ax1.set_ylabel('Average Tempo (BPM)', fontsize=14, color=color1)
    ax1.plot(tempo_by_decade.index, tempo_by_decade.values, marker='o', 
             linewidth=3, markersize=10, color=color1, label='Tempo')
    ax1.tick_params(axis='y', labelcolor=color1)
    
    # Create second y-axis for valence
    ax2 = ax1.twinx()
    ax2.set_ylabel('Average Valence (0-1)', fontsize=14, color=color2)
    ax2.plot(valence_by_decade.index, valence_by_decade.values, marker='s', 
             linewidth=3, markersize=10, color=color2, label='Valence')
    ax2.tick_params(axis='y', labelcolor=color2)
    
    # Add grid and title
    ax1.grid(True, alpha=0.3)
    plt.title('Tempo and Valence Trends by Decade', fontsize=18)
    
    # Add combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper center', 
               bbox_to_anchor=(0.5, -0.1), ncol=2, fontsize=12)
    
    plt.tight_layout()
    plt.show()

# Create heatmaps and timelines
create_heatmaps_and_timelines(df)